# Memory Management and Performance Tuning

Deep dive into Spark's memory management, storage levels, and performance optimization techniques.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.storagelevel import StorageLevel
from pyspark import SparkConf
import time
import psutil
import matplotlib.pyplot as plt
import numpy as np

# Configure Spark for memory management demonstration
conf = SparkConf()
conf.set("spark.executor.memory", "2g")
conf.set("spark.executor.memoryFraction", "0.8")
conf.set("spark.storage.memoryFraction", "0.5")
conf.set("spark.sql.execution.arrow.maxRecordsPerBatch", "10000")

spark = SparkSession.builder \
    .appName("MemoryManagementDemo") \
    .master("spark://spark-master:7077") \
    .config(conf=conf) \
    .getOrCreate()

print(f"Spark Version: {spark.version}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")

## 1. Understanding Spark Memory Model

In [ ]:
print("=== SPARK MEMORY MODEL ===")

# Get memory configuration
executor_memory = spark.conf.get("spark.executor.memory")
memory_fraction = float(spark.conf.get("spark.executor.memoryFraction", "0.6"))
storage_fraction = float(spark.conf.get("spark.storage.memoryFraction", "0.5"))

print(f"Executor Memory: {executor_memory}")
print(f"Memory Fraction: {memory_fraction}")
print(f"Storage Memory Fraction: {storage_fraction}")

# Calculate memory regions
total_memory_mb = 2048  # 2GB in MB
jvm_overhead = total_memory_mb * 0.1
spark_memory = total_memory_mb * memory_fraction
storage_memory = spark_memory * storage_fraction
execution_memory = spark_memory * (1 - storage_fraction)
user_memory = total_memory_mb - jvm_overhead - spark_memory

print(f"\nMemory Breakdown (MB):")
print(f"Total Memory: {total_memory_mb}")
print(f"JVM Overhead: {jvm_overhead}")
print(f"User Memory: {user_memory}")
print(f"Storage Memory: {storage_memory}")
print(f"Execution Memory: {execution_memory}")

# Visualize memory layout
labels = ['JVM Overhead', 'User Memory', 'Storage Memory', 'Execution Memory']
sizes = [jvm_overhead, user_memory, storage_memory, execution_memory]
colors = ['#ff9999', '#66b3ff', '#99ff99', '#ffcc99']

plt.figure(figsize=(10, 6))
plt.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
plt.title('Spark Memory Model Distribution')
plt.axis('equal')
plt.show()

## 2. Storage Levels Comparison

In [ ]:
print("=== STORAGE LEVELS COMPARISON ===")

# Create a test dataset
test_data = []
for i in range(50000):
    test_data.append((
        i,
        f"user_{i % 1000}",
        np.random.uniform(10, 1000),
        np.random.choice(['A', 'B', 'C', 'D']),
        f"2024-{(i % 12) + 1:02d}-{(i % 28) + 1:02d}"
    ))

schema = "id INT, user_id STRING, amount DOUBLE, category STRING, date STRING"
test_df = spark.createDataFrame(test_data, schema)

print(f"Test dataset size: {test_df.count()} records")

# Function to measure performance with different storage levels
def test_storage_level(df, storage_level, level_name):
    print(f"\n--- Testing {level_name} ---")
    
    # Clear any existing cache
    df.unpersist()
    
    # Cache with specific storage level
    df.persist(storage_level)
    
    # First action to trigger caching
    start_time = time.time()
    count1 = df.count()
    cache_time = time.time() - start_time
    
    # Second action to test cache hit
    start_time = time.time()
    count2 = df.groupBy("category").sum("amount").count()
    hit_time = time.time() - start_time
    
    # Third action for aggregation
    start_time = time.time()
    avg_amount = df.agg(avg("amount")).collect()[0][0]
    agg_time = time.time() - start_time
    
    print(f"Cache Time: {cache_time:.3f}s")
    print(f"Cache Hit Time: {hit_time:.3f}s")
    print(f"Aggregation Time: {agg_time:.3f}s")
    print(f"Count: {count1}, Avg Amount: {avg_amount:.2f}")
    
    # Get storage info
    storage_level_str = str(storage_level)
    print(f"Storage Level: {storage_level_str}")
    
    return cache_time, hit_time, agg_time

# Test different storage levels
storage_levels = [
    (StorageLevel.MEMORY_ONLY, "MEMORY_ONLY"),
    (StorageLevel.MEMORY_AND_DISK, "MEMORY_AND_DISK"),
    (StorageLevel.MEMORY_ONLY_SER, "MEMORY_ONLY_SER"),
    (StorageLevel.MEMORY_AND_DISK_SER, "MEMORY_AND_DISK_SER"),
    (StorageLevel.DISK_ONLY, "DISK_ONLY")
]

results = []
for storage_level, level_name in storage_levels:
    cache_time, hit_time, agg_time = test_storage_level(test_df, storage_level, level_name)
    results.append((level_name, cache_time, hit_time, agg_time))

# Visualize results
level_names = [r[0] for r in results]
cache_times = [r[1] for r in results]
hit_times = [r[2] for r in results]
agg_times = [r[3] for r in results]

x = np.arange(len(level_names))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(x - width, cache_times, width, label='Cache Time', color='skyblue')
ax.bar(x, hit_times, width, label='Cache Hit Time', color='lightgreen')
ax.bar(x + width, agg_times, width, label='Aggregation Time', color='salmon')

ax.set_xlabel('Storage Level')
ax.set_ylabel('Time (seconds)')
ax.set_title('Performance Comparison of Storage Levels')
ax.set_xticks(x)
ax.set_xticklabels(level_names, rotation=45)
ax.legend()

plt.tight_layout()
plt.show()

# Cleanup
test_df.unpersist()

## 3. Memory Usage Monitoring

In [ ]:
print("=== MEMORY USAGE MONITORING ===")

def get_memory_info():
    """Get current memory information"""
    status_tracker = spark.sparkContext.statusTracker()
    executor_infos = status_tracker.getExecutorInfos()
    
    total_memory = 0
    used_memory = 0
    
    print("Executor Memory Information:")
    for executor in executor_infos:
        total_memory += executor.maxMemory
        used_memory += executor.memoryUsed
        
        print(f"Executor {executor.executorId}:")
        print(f"  Max Memory: {executor.maxMemory / 1024 / 1024:.1f} MB")
        print(f"  Used Memory: {executor.memoryUsed / 1024 / 1024:.1f} MB")
        print(f"  Disk Used: {executor.diskUsed / 1024 / 1024:.1f} MB")
        print(f"  Active Tasks: {executor.activeTasks}")
        print(f"  Total Tasks: {executor.totalTasks}")
    
    utilization = (used_memory / total_memory * 100) if total_memory > 0 else 0
    print(f"\nOverall Memory Utilization: {utilization:.1f}%")
    
    return total_memory, used_memory

# Initial memory state
print("Initial Memory State:")
initial_total, initial_used = get_memory_info()

# Create and cache multiple DataFrames to observe memory usage
print("\n=== Creating and Caching DataFrames ===")

# Load and cache sales data
sales_df = spark.read.option("header", "true").option("inferSchema", "true").csv("/home/jovyan/data/sales_data.csv")
sales_df.cache()
sales_count = sales_df.count()
print(f"Sales DF cached: {sales_count} records")

print("\nMemory after caching sales:")
sales_total, sales_used = get_memory_info()

# Create and cache a larger synthetic dataset
large_data = []
for i in range(100000):
    large_data.append((
        i,
        f"product_{i % 1000}",
        np.random.uniform(1, 500),
        np.random.choice(['Electronics', 'Clothing', 'Books', 'Sports']),
        np.random.choice(['North', 'South', 'East', 'West'])
    ))

large_df = spark.createDataFrame(large_data, "id INT, product STRING, price DOUBLE, category STRING, region STRING")
large_df.cache()
large_count = large_df.count()
print(f"\nLarge DF cached: {large_count} records")

print("\nMemory after caching large dataset:")
large_total, large_used = get_memory_info()

# Show RDD storage information
print("\n=== RDD Storage Information ===")
rdd_infos = spark.sparkContext.statusTracker().getRDDInfos()
for rdd_info in rdd_infos:
    if rdd_info.numPartitions > 0:
        print(f"RDD {rdd_info.id}: {rdd_info.name}")
        print(f"  Partitions: {rdd_info.numPartitions}")
        print(f"  Storage Level: {rdd_info.storageLevel}")
        print(f"  Memory Size: {rdd_info.memorySize / 1024 / 1024:.1f} MB")
        print(f"  Disk Size: {rdd_info.diskSize / 1024 / 1024:.1f} MB")

## 4. Garbage Collection Tuning

In [ ]:
print("=== GARBAGE COLLECTION TUNING ===")

# Function to demonstrate GC impact
def gc_stress_test(iterations=5):
    """Create and destroy DataFrames to trigger GC"""
    times = []
    
    for i in range(iterations):
        # Create temporary large dataset
        temp_data = [(j, f"temp_{j}", np.random.uniform(1, 100)) for j in range(20000)]
        temp_df = spark.createDataFrame(temp_data, "id INT, name STRING, value DOUBLE")
        
        start_time = time.time()
        
        # Perform operations
        result = temp_df.groupBy("name").agg(sum("value")).count()
        
        elapsed = time.time() - start_time
        times.append(elapsed)
        
        print(f"Iteration {i+1}: {elapsed:.3f}s, Result: {result}")
        
        # Force cleanup
        temp_df.unpersist()
        del temp_df
    
    avg_time = np.mean(times)
    std_time = np.std(times)
    
    print(f"\nAverage time: {avg_time:.3f}s ± {std_time:.3f}s")
    
    return times

# Run GC stress test
gc_times = gc_stress_test()

# Visualize GC impact
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(gc_times) + 1), gc_times, 'bo-', linewidth=2, markersize=8)
plt.xlabel('Iteration')
plt.ylabel('Execution Time (seconds)')
plt.title('GC Impact on Performance')
plt.grid(True, alpha=0.3)
plt.axhline(y=np.mean(gc_times), color='r', linestyle='--', label=f'Average: {np.mean(gc_times):.3f}s')
plt.legend()
plt.show()

print("\n=== GC TUNING RECOMMENDATIONS ===")
recommendations = [
    "1. Use G1GC for better pause times: -XX:+UseG1GC",
    "2. Set appropriate heap size: -Xmx and -Xms",
    "3. Tune GC pause target: -XX:MaxGCPauseMillis=200",
    "4. Monitor GC logs: -XX:+PrintGC -XX:+PrintGCDetails",
    "5. Use off-heap storage for large datasets",
    "6. Increase parallelism to reduce per-task memory",
    "7. Use serialized storage levels for memory efficiency"
]

for rec in recommendations:
    print(rec)

## 5. Performance Tuning Techniques

In [ ]:
print("=== PERFORMANCE TUNING TECHNIQUES ===")

# Load test data
sales_df = spark.read.option("header", "true").option("inferSchema", "true").csv("/home/jovyan/data/sales_data.csv")
customers_df = spark.read.option("header", "true").option("inferSchema", "true").csv("/home/jovyan/data/customers.csv")

# 1. Broadcast Variables for Lookup Tables
print("1. Broadcast Variables Demo:")

# Create a lookup dictionary
category_mapping = {
    'Electronics': 'TECH',
    'Clothing': 'APPAREL',
    'Books': 'MEDIA',
    'Sports': 'FITNESS'
}

# Broadcast the mapping
broadcast_mapping = spark.sparkContext.broadcast(category_mapping)

def map_category(category):
    return broadcast_mapping.value.get(category, 'OTHER')

map_category_udf = udf(map_category, StringType())

# Apply broadcast variable
mapped_sales = sales_df.withColumn("category_code", map_category_udf(col("category")))
mapped_sales.select("category", "category_code").distinct().show()

# 2. Partitioning Optimization
print("\n2. Partitioning Optimization:")

# Compare different partitioning strategies
def test_partitioning_performance(df, partition_strategy, strategy_name):
    print(f"\nTesting {strategy_name}:")
    
    start_time = time.time()
    result = partition_strategy(df).groupBy("region").agg(
        sum("amount").alias("total_sales"),
        count("*").alias("transaction_count")
    ).collect()
    elapsed = time.time() - start_time
    
    print(f"Execution time: {elapsed:.3f}s")
    print(f"Partitions: {partition_strategy(df).rdd.getNumPartitions()}")
    
    return elapsed

# Create larger dataset for meaningful comparison
large_sales_data = []
for i in range(10000):
    large_sales_data.append((
        i + 1000,
        (i % 1000) + 1,
        (i % 100) + 1,
        np.random.choice(['Electronics', 'Clothing', 'Books', 'Sports']),
        round(np.random.uniform(10, 2000), 2),
        f"2024-01-{(i % 30) + 1:02d} 10:00:00",
        np.random.choice(['North', 'South', 'East', 'West'])
    ))

large_sales_df = spark.createDataFrame(large_sales_data, sales_df.schema)

# Test different partitioning strategies
strategies = [
    (lambda df: df, "Default Partitioning"),
    (lambda df: df.repartition(8), "8 Partitions"),
    (lambda df: df.repartition("region"), "Partition by Region"),
    (lambda df: df.repartition(4, "region"), "4 Partitions by Region")
]

partition_times = []
for strategy_func, strategy_name in strategies:
    elapsed = test_partitioning_performance(large_sales_df, strategy_func, strategy_name)
    partition_times.append(elapsed)

# 3. Cache vs Persist Performance
print("\n3. Cache vs Persist Performance:")

def test_cache_performance():
    # Test without caching
    large_sales_df.unpersist()
    start_time = time.time()
    result1 = large_sales_df.groupBy("category").sum("amount").count()
    result2 = large_sales_df.groupBy("region").avg("amount").count()
    no_cache_time = time.time() - start_time
    
    # Test with caching
    large_sales_df.cache()
    start_time = time.time()
    result1 = large_sales_df.groupBy("category").sum("amount").count()
    result2 = large_sales_df.groupBy("region").avg("amount").count()
    cache_time = time.time() - start_time
    
    print(f"Without cache: {no_cache_time:.3f}s")
    print(f"With cache: {cache_time:.3f}s")
    print(f"Speedup: {no_cache_time / cache_time:.2f}x")
    
    return no_cache_time, cache_time

no_cache_time, cache_time = test_cache_performance()

# Visualize performance comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Partitioning performance
strategy_names = [s[1] for s in strategies]
ax1.bar(range(len(partition_times)), partition_times, color='lightblue')
ax1.set_xlabel('Partitioning Strategy')
ax1.set_ylabel('Execution Time (seconds)')
ax1.set_title('Partitioning Strategy Performance')
ax1.set_xticks(range(len(strategy_names)))
ax1.set_xticklabels(strategy_names, rotation=45, ha='right')

# Cache performance
cache_comparison = ['No Cache', 'With Cache']
cache_times = [no_cache_time, cache_time]
ax2.bar(cache_comparison, cache_times, color=['lightcoral', 'lightgreen'])
ax2.set_ylabel('Execution Time (seconds)')
ax2.set_title('Cache Performance Comparison')

plt.tight_layout()
plt.show()

# Cleanup
large_sales_df.unpersist()
broadcast_mapping.unpersist()

## 6. Memory Optimization Best Practices

In [ ]:
print("=== MEMORY OPTIMIZATION BEST PRACTICES ===")

best_practices = {
    "Storage Level Selection": [
        "Use MEMORY_ONLY for frequently accessed small datasets",
        "Use MEMORY_AND_DISK for large datasets with reuse",
        "Use serialized levels (SER) for memory-constrained environments",
        "Use DISK_ONLY for large datasets accessed infrequently"
    ],
    "Memory Configuration": [
        "Set spark.executor.memory appropriately (typically 80% of available memory)",
        "Tune spark.executor.memoryFraction (default 0.6)",
        "Adjust spark.storage.memoryFraction (default 0.5)",
        "Configure spark.executor.instances based on cluster size"
    ],
    "Data Management": [
        "Use columnar formats (Parquet) for better compression",
        "Partition data by frequently filtered columns",
        "Use broadcast variables for small lookup tables",
        "Unpersist DataFrames when no longer needed"
    ],
    "GC Optimization": [
        "Use G1GC for better pause times",
        "Set -XX:MaxGCPauseMillis=200 for low latency",
        "Monitor GC logs to identify issues",
        "Consider off-heap storage for large caches"
    ],
    "Code Optimization": [
        "Avoid using collect() on large datasets",
        "Use efficient data types (avoid strings when possible)",
        "Minimize object creation in UDFs",
        "Use built-in functions instead of UDFs when possible"
    ]
}

for category, practices in best_practices.items():
    print(f"\n📋 {category}:")
    for i, practice in enumerate(practices, 1):
        print(f"  {i}. {practice}")

# Memory monitoring function
def memory_monitoring_summary():
    print("\n=== MEMORY MONITORING CHECKLIST ===")
    
    checklist = [
        "Monitor Spark UI Storage tab for cached data",
        "Check Executors tab for memory utilization",
        "Watch for spill metrics in task details",
        "Monitor GC time in executor logs",
        "Use external monitoring tools (Ganglia, Grafana)",
        "Set up alerts for memory pressure",
        "Track memory usage trends over time"
    ]
    
    for i, item in enumerate(checklist, 1):
        print(f"  ☐ {item}")

memory_monitoring_summary()

print("\n🎯 FINAL MEMORY OPTIMIZATION SCORE CARD:")
print("=" * 50)
print("✅ Demonstrated multiple storage levels")
print("✅ Compared caching strategies")
print("✅ Showed memory monitoring techniques")
print("✅ Provided GC tuning guidance")
print("✅ Listed optimization best practices")
print("🚀 Ready for production memory optimization!")